<a href="https://colab.research.google.com/github/MHamza-Ahmad/Flyrank-Internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MHamza-Ahmad/Flyrank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
rule_name = "Forecasted Missed Clicks (Page 1 Underperformers)"
target_ctr = 0.05
threshold_ctr = 0.03
max_position = 10

print(f"Active Rule: {rule_name}")
print(f"Targeting: Position <= {max_position} AND CTR < {threshold_ctr}")

Active Rule: Forecasted Missed Clicks (Page 1 Underperformers)
Targeting: Position <= 10 AND CTR < 0.03


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import os

print("Generating synthetic Search Console data for code verification...")

# 1. Generate Dataset
np.random.seed(42)
n_rows = 5000
df = pd.DataFrame({
    'query': [f'query_{i}' for i in range(n_rows)],
    'url': [f'/page_{i%50}' for i in range(n_rows)],
    'impressions': np.random.exponential(scale=1000, size=n_rows).astype(int) + 10,
    'position': np.random.uniform(1, 50, size=n_rows)
})

# Calculate clicks and CTR
base_ctr = np.where(df['position'] <= 10, 0.04, 0.01)
df['clicks'] = (df['impressions'] * np.clip(base_ctr + np.random.normal(0, 0.02, n_rows), 0, 1)).astype(int)
df['ctr'] = (df['clicks'] / df['impressions']).fillna(0)

# 2. Apply Rule & Calculate Score
df['forecasted_missed_clicks'] = np.where(
    (df['position'] <= 10) & (df['ctr'] < 0.03),
    (df['impressions'] * 0.05) - df['clicks'],
    0
)

# 3. Assign Reason Codes and Actions
df['reason_code'] = np.where(df['forecasted_missed_clicks'] > 0, 'PAGE_1_UNDERPERFORMER', 'NO_ACTION_REQUIRED')
df['action_label'] = np.where(df['reason_code'] == 'PAGE_1_UNDERPERFORMER', 'REWRITE_TITLE_AND_META', 'MAINTAIN')

# 4. Filter and Rank
queue = df[df['action_label'] != 'MAINTAIN'].copy()
queue = queue.sort_values(by='forecasted_missed_clicks', ascending=False)

output_cols = ['query', 'url', 'impressions', 'position', 'ctr', 'forecasted_missed_clicks', 'reason_code', 'action_label']
final_queue = queue[output_cols].head(100) # Keep top 100 for the file

# 5. Save to outputs folder
os.makedirs('work/outputs', exist_ok=True)
csv_path = 'work/outputs/baseline_action_score.csv'
final_queue.to_csv(csv_path, index=False)

print(f"Ranked queue generated with {len(queue)} actionable rows.")
print(f"File saved to: {csv_path}")

Generating synthetic Search Console data for code verification...
Ranked queue generated with 293 actionable rows.
File saved to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
print("--- TOP 20 QUEUE FOR REVIEW ---")
display(final_queue.head(20))

--- TOP 20 QUEUE FOR REVIEW ---


,query,url,impressions,position,ctr,forecasted_missed_clicks,reason_code,action_label
2314,query_2314,/page_14,4946,4.253862,0.001617,239.30,PAGE_1_UNDERPERFORMER,REWRITE_TITLE_AND_META
1638,query_1638,/page_38,4900,5.184722,0.014082,176.00,PAGE_1_UNDERPERFORMER,REWRITE_TITLE_AND_META
3924,query_3924,/page_24,4648,8.717527,0.017427,151.40,PAGE_1_UNDERPERFORMER,REWRITE_TITLE_AND_META
1324,query_1324,/page_24,3734,4.992221,0.009909,149.70,PAGE_1_UNDERPERFORMER,REWRITE_TITLE_AND_META
3882,query_3882,/page_32,3182,2.674604,0.003143,149.10,PAGE_1_UNDERPERFORMER,REWRITE_TITLE_AND_META
2638,query_2638,/page_38,3021,3.760331,0.003310,141.05,PAGE_1_UNDERPERFORMER,REWRITE_TITLE_AND_META
4509,query_4509,/page_9,2848,8.855143,0.002458,135.40,PAGE_1_UNDERPERFORMER,REWRITE_TITLE_AND_META
1773,query_1773,/page_23,3118,9.071814,0.007697,131.90,PAGE_1_UNDERPERFORMER,REWRITE_TITLE_AND_META
4404,query_4404,/page_4,4566,9.189581,0.023215,122.30,PAGE_1_UNDERPERFORMER,REWRITE_TITLE_AND_META
1360,query_1360,/page_10,2444,4.180065,0.001227,119.20,PAGE_1_UNDERPERFORMER,REWRITE_TITLE_AND_META


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
Weak Picks: The most obvious weak picks in this queue are queries sitting at position 9 or 10. A CTR of 1.5% to 2% at the bottom of Page 1 is actually average performance. Flagging them as "underperformers" based on a static 5% target CTR highlights the limitation of this hardcoded baseline. A true ML model will need to learn the natural CTR decay curve based on position.

Leakage Check: I have verified that no target-derived labels (like future_clicks) or proprietary product flags were used to generate this score. The score relies purely on impressions, position, and ctr available on the day of analysis.

Python

In [4]:
banned_columns = ['future_clicks', 'next_month_ctr', 'flyrank_flag_active']
leak_check = [col for col in banned_columns if col in df.columns]

assert len(leak_check) == 0, f"LEAKAGE DETECTED: {leak_check}"
print("Leakage check passed: No future windows or product flags detected in feature set.")

# Weak pick identification: Count how many of our top 20 are at position 9 or 10
weak_picks = final_queue.head(20)[final_queue.head(20)['position'] >= 9.0]
print(f"Identified {len(weak_picks)} potentially weak picks in the Top 20 (Position >= 9).")

Leakage check passed: No future windows or product flags detected in feature set.
Identified 3 potentially weak picks in the Top 20 (Position >= 9).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.